# ⚡ Economic Load Dispatch with Prohibited Operating Zones

> 🚀 **Don't choke your local machine.** Qoro is giving away **$100 in free cloud compute credits.**
> Get your API key at **[dash.qoroquantum.net](https://dash.qoroquantum.net)** to run this at scale.

## Why Cloud?

The PCE-VQE optimizer needs **hundreds of circuit evaluations** per iteration. Running these sequentially
on a local simulator means every evaluation blocks the next. QoroService evaluates circuits **in parallel**,
collapsing wall-clock time dramatically.

## Step 0: Get Your API Key

Create a `.env` file in the repo root:
```
QORO_API_KEY="your_api_key_here"
```

👉 **[Get your free API key →](https://dash.qoroquantum.net)**

## Phase 1 — Local Toy Problem (3 generators, 12 binary vars)

In [ ]:
import dimod
import pennylane as qml
import time

from divi.backends import ParallelSimulator, QoroService, JobConfig
from divi.qprog import PCE, GenericLayerAnsatz
from divi.qprog.optimizers import PymooMethod, PymooOptimizer
from divi.typing import qubo_to_matrix

### Step 1 — Define the generators

Each generator has a quadratic fuel cost curve, operating limits, and a
Prohibited Operating Zone (vibration band) that must be avoided.

In [ ]:
STEP_MW = 5
N_QUBITS_PER_GEN = 4
BIT_WEIGHTS = [2**b for b in range(N_QUBITS_PER_GEN)]
DEMAND = 195  # MW

def fuel_cost(gen, power):
    return gen["a"] + gen["b"] * power + gen["c"] * power ** 2

def _qubit_name(gen_idx, bit_idx):
    return f"q_{gen_idx}_{bit_idx}"

generators = [
    {"name": "Gen 1", "a": 20, "b": 2.0, "c": 0.010,
     "P_min": 40, "P_max": 115, "poz_low": 60, "poz_high": 75},
    {"name": "Gen 2", "a": 15, "b": 1.5, "c": 0.020,
     "P_min": 20, "P_max": 95, "poz_low": 40, "poz_high": 55},
    {"name": "Gen 3", "a": 25, "b": 1.8, "c": 0.015,
     "P_min": 30, "P_max": 105, "poz_low": 50, "poz_high": 65},
]

print(f"3 generators, demand = {DEMAND} MW")
for g in generators:
    print(f"  {g['name']}: {g['P_min']}–{g['P_max']} MW, POZ [{g['poz_low']},{g['poz_high']}]")

### Step 2 — Build the QUBO

In [ ]:
def build_qubo(generators, demand, penalty_lambda=2000, poz_mu=5000):
    bqm = dimod.BinaryQuadraticModel(vartype="BINARY")
    var_names = []
    for g in range(len(generators)):
        for b in range(N_QUBITS_PER_GEN):
            var_names.append(_qubit_name(g, b))
    cost_offset = 0.0
    for g, gen in enumerate(generators):
        a, b_coeff, c = gen["a"], gen["b"], gen["c"]
        p_min = gen["P_min"]
        cost_offset += a + b_coeff * p_min + c * p_min ** 2
        for k in range(N_QUBITS_PER_GEN):
            w_k = STEP_MW * BIT_WEIGHTS[k]
            var_k = _qubit_name(g, k)
            bqm.add_linear(var_k, b_coeff * w_k + c * (2 * p_min * w_k + w_k ** 2))
            for l in range(k + 1, N_QUBITS_PER_GEN):
                w_l = STEP_MW * BIT_WEIGHTS[l]
                bqm.add_quadratic(var_k, _qubit_name(g, l), c * 2 * w_k * w_l)
    bqm.offset += cost_offset
    d_const = sum(gen["P_min"] for gen in generators) - demand
    d_terms = []
    for g in range(len(generators)):
        for k in range(N_QUBITS_PER_GEN):
            w = STEP_MW * BIT_WEIGHTS[k]
            d_terms.append((_qubit_name(g, k), w))
    for var_i, w_i in d_terms:
        bqm.add_linear(var_i, penalty_lambda * (2 * d_const * w_i + w_i ** 2))
    for i in range(len(d_terms)):
        vi, wi = d_terms[i]
        for j in range(i + 1, len(d_terms)):
            vj, wj = d_terms[j]
            bqm.add_quadratic(vi, vj, penalty_lambda * 2 * wi * wj)
    for g in range(len(generators)):
        q_msb = _qubit_name(g, 3)
        q_2nd = _qubit_name(g, 2)
        bqm.add_linear(q_2nd, poz_mu)
        bqm.add_quadratic(q_msb, q_2nd, -poz_mu)
    return bqm, var_names

bqm, var_names = build_qubo(generators, demand=DEMAND)
print(f"Built QUBO: {len(var_names)} variables, {len(bqm.quadratic)} interactions")

### Step 3 — Solve with PCE-VQE (locally)

In [ ]:
def decode_power(gen_idx, gens, bit_values):
    integer_val = sum(BIT_WEIGHTS[b] * bit_values[_qubit_name(gen_idx, b)] for b in range(N_QUBITS_PER_GEN))
    return gens[gen_idx]["P_min"] + STEP_MW * integer_val

def repair_solution(powers, gens, demand):
    for g, gen in enumerate(gens):
        if gen["poz_low"] <= powers[g] <= gen["poz_high"]:
            if abs(powers[g] - gen["poz_low"]) <= abs(powers[g] - gen["poz_high"]):
                powers[g] = gen["poz_low"] - STEP_MW
            else:
                powers[g] = gen["poz_high"] + STEP_MW
            powers[g] = max(gen["P_min"], min(gen["P_max"], powers[g]))
    deficit = demand - sum(powers)
    for g in sorted(range(len(gens)), key=lambda g: gens[g]["b"]):
        gen = gens[g]
        headroom = gen["P_max"] - powers[g]
        adjust = max(-powers[g] + gen["P_min"], min(headroom, deficit))
        powers[g] += adjust
        deficit -= adjust
        if abs(deficit) < 0.1:
            break
    return powers

backend = ParallelSimulator(shots=10_000)
qubo_mat = qubo_to_matrix(bqm)
ansatz = GenericLayerAnsatz(gate_sequence=[qml.RY, qml.RZ], entangler=qml.CNOT, entangling_layout="all-to-all")

t0 = time.time()
pce_solver = PCE(
    qubo_mat, ansatz=ansatz, n_layers=3, encoding_type="poly",
    optimizer=PymooOptimizer(method=PymooMethod.DE, population_size=30),
    max_iterations=20, alpha=3.0, backend=backend,
)
print(f"PCE qubits: {pce_solver.n_qubits} (poly encoding of {len(var_names)} variables)")
pce_solver.run()
local_time = time.time() - t0
print(f"✅ Phase 1 solved in {local_time:.1f}s")

### Step 4 — Decode and repair

In [ ]:
top_solutions = pce_solver.get_top_solutions(n=20, include_decoded=True)
var_list = list(bqm.variables)

best = None
for sol in top_solutions:
    sample = {var_list[k]: int(sol.decoded[k]) for k in range(len(var_list))}
    powers = [decode_power(g, generators, sample) for g in range(len(generators))]
    repaired = repair_solution(powers, generators, DEMAND)
    rep_cost = sum(fuel_cost(gen, p) for gen, p in zip(generators, repaired))
    if best is None or rep_cost < best[1]:
        best = (repaired, rep_cost, sol.prob)

if best:
    powers, cost, prob = best
    gen_str = ", ".join(f"P{i+1}={p:.0f}" for i, p in enumerate(powers))
    print(f"Best repaired solution: {gen_str} MW")
    print(f"Total={sum(powers):.0f} MW, Cost={cost:.1f} $, quantum seed prob={prob:.2%}")

---

## Phase 2 — Scale Up to 6 Generators with QoroService

Double the generators: **24 binary variables → ~7 PCE qubits**.
QoroService parallelises all circuit evaluations on Maestro.

In [ ]:
generators_large = generators + [
    {"name": "Gen 4", "a": 18, "b": 1.7, "c": 0.012,
     "P_min": 35, "P_max": 110, "poz_low": 55, "poz_high": 70},
    {"name": "Gen 5", "a": 22, "b": 2.2, "c": 0.009,
     "P_min": 50, "P_max": 120, "poz_low": 65, "poz_high": 80},
    {"name": "Gen 6", "a": 12, "b": 1.3, "c": 0.018,
     "P_min": 25, "P_max": 90, "poz_low": 45, "poz_high": 60},
]
DEMAND_LARGE = 390

bqm_large, var_names_large = build_qubo(generators_large, demand=DEMAND_LARGE)
print(f"Built QUBO: {len(var_names_large)} variables (was {len(var_names)}) — {len(bqm_large.quadratic)} interactions")

In [ ]:
cloud_backend = QoroService(job_config=JobConfig(shots=10_000))
qubo_mat_large = qubo_to_matrix(bqm_large)
ansatz_large = GenericLayerAnsatz(gate_sequence=[qml.RY, qml.RZ], entangler=qml.CNOT, entangling_layout="all-to-all")

t0 = time.time()
pce_cloud = PCE(
    qubo_mat_large, ansatz=ansatz_large, n_layers=3, encoding_type="poly",
    optimizer=PymooOptimizer(method=PymooMethod.DE, population_size=30),
    max_iterations=10, alpha=3.0, backend=cloud_backend,
)
print(f"PCE qubits: {pce_cloud.n_qubits} (poly encoding of {len(var_names_large)} variables)")
pce_cloud.run()
cloud_time = time.time() - t0
print(f"✅ Phase 2 solved in {cloud_time:.1f}s")

In [ ]:
top_cloud = pce_cloud.get_top_solutions(n=20, include_decoded=True)
var_list_large = list(bqm_large.variables)

best_cloud = None
for sol in top_cloud:
    sample = {var_list_large[k]: int(sol.decoded_bitstring[k]) for k in range(len(var_list_large))}
    powers = [decode_power(g, generators_large, sample) for g in range(len(generators_large))]
    repaired = repair_solution(powers, generators_large, DEMAND_LARGE)
    rep_cost = sum(fuel_cost(gen, p) for gen, p in zip(generators_large, repaired))
    if best_cloud is None or rep_cost < best_cloud[1]:
        best_cloud = (repaired, rep_cost, sol.prob)

if best_cloud:
    powers, cost, prob = best_cloud
    gen_str = ", ".join(f"P{i+1}={p:.0f}" for i, p in enumerate(powers))
    print(f"Best repaired solution: {gen_str} MW")
    print(f"Total={sum(powers):.0f} MW, Cost={cost:.1f} $, quantum seed prob={prob:.2%}")

print(f"⚡ Cloud  (Phase 2): {cloud_time:.1f}s for 6 generators ({len(var_names_large)} vars)")

---

## 🎉 That's the Power of QoroService

Your laptop handled **3 generators** (12 vars). Qoro Maestro handled **6 generators** (24 vars) —
double the problem size with parallel circuit evaluation.

👉 **[Get your API key and start scaling →](https://dash.qoroquantum.net)**